# 01 - Dataset Analysis

This notebook inspects a small **SYNTHETIC DEMO DATA** sample generated by the same loader used by the experiment suite. It is a demonstration, not a real utility dataset.

Run from the project root or from `notebooks/`.

In [ ]:
from pathlib import Path
import json
import sys

# Headless-safe plotting; Jupyter still renders figures inline.
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


## Generate a small labelled sample

The full quick run uses 240 consumers x 180 days. Here we use 60 consumers x 90 days so the notebook is fast.

In [ ]:
import pandas as pd
from utilities.config import load_config
from preprocessing.loader import generate_synthetic_dataset

cfg = load_config("quick", [
    "data.synthetic.consumers=60",
    "data.synthetic.days=90",
    "data.synthetic.theft_rate=0.15",
])
bundle = generate_synthetic_dataset(cfg, seed=42)
print("Source:", bundle.source)
print("Synthetic:", bundle.is_synthetic)
print(json.dumps(bundle.stats, indent=2))
labels = bundle.labels.astype(int)
print(labels.value_counts().rename(index={0: "normal", 1: "theft"}))


## Raw rows

The loader returns a tidy long table: consumer, timestamp, consumption. Labels are per consumer.

In [ ]:
display(bundle.long_df.head())  # noqa: F821 - display exists in Jupyter
print("Rows:", len(bundle.long_df), "| Consumers:", len(bundle.labels))
print("Consumers with theft label:", int(labels.sum()))


## Mean daily consumption by consumer label

The synthetic theft mechanism suppresses metered consumption after a random activation day, so the gap is descriptive rather than a classifier.

In [ ]:
import matplotlib.pyplot as plt

long = bundle.long_df.copy()
long["label"] = long["consumer_id"].astype(str).map(labels)
long["date"] = pd.to_datetime(long["timestamp"]).dt.date
daily = long.groupby(["date", "label"])["consumption"].mean().unstack()
daily.columns = ["normal", "theft"]
daily.plot(figsize=(10, 3), title="SYNTHETIC DEMO DATA: mean daily consumption")
plt.ylabel("consumption (scaled units)")
plt.tight_layout()
plt.show()


## Current project result summary (if present)

In [ ]:
summary_path = ROOT / "results" / "metrics" / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Suite:", summary.get("suite_id"))
    print("Mode:", summary.get("mode"), "| Demo:", summary.get("demo"), "| Seeds:", summary.get("seeds"))
    display(pd.DataFrame(summary.get("dataset", {}).get("table", [])))  # noqa: F821
else:
    print("No summary yet. Run: python main.py --demo or python main.py --mode quick")
